In [ ]:
import json
import pandas as pd

results = json.load(open('eval_results.json'))
report = json.load(open('experiment_report.json'))

## Per-Sample Predictions

In [ ]:
df = pd.DataFrame(results['samples'])
df['filename'] = df['filepath'].apply(lambda p: p.split('/')[-1])
df['correct'] = df['ground_truth'] == df['pegasus_label']
df[['filename', 'ground_truth', 'pegasus_label', 'correct', 'embedding_dim']]

## Metrics Summary

In [ ]:
pd.DataFrame([results['metrics']]).T.rename(columns={0: 'value'})

## Retrieval Per-Query Breakdown

In [ ]:
retrieval = report['results']['benchmark_b_retrieval_quality']['per_query']
rdf = pd.DataFrame([
    {'query': q, 'expected': ', '.join(v['expected']), 'AP@5': v['ap'], 'Recall@5': v['recall']}
    for q, v in retrieval.items()
]).sort_values('AP@5', ascending=False)
rdf

## Pegasus Per-Class Breakdown

In [ ]:
per_class = report['results']['benchmark_c_generation_quality']['per_class_results']
pcdf = pd.DataFrame(per_class).T
pcdf.index.name = 'class'
pcdf

## Pegasus Output Distribution

In [ ]:
df['pegasus_label'].value_counts().plot.barh(title='Pegasus Output Distribution')

## Confusion: Ground Truth vs Pegasus

In [ ]:
labeled = df[~df['pegasus_label'].isin(['ERROR', None])]
pd.crosstab(labeled['ground_truth'], labeled['pegasus_label'], margins=True)